[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-10-lora-from-scratch.ipynb)

# Advanced Discussion 10 — LoRA from scratch

A LoRA layer, merge check, parameter arithmetic, a rank sweep against full fine-tuning and last-layer tuning, and the SVD of a fine-tuning update.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Uses scikit-learn's bundled digits; runs in about a minute.

In [ ]:
import copy, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0); np.random.seed(0)

## 1. LoRA from scratch: W x + (alpha/r) * B A x, with W frozen

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__(); self.base = base; self.scale = alpha / r
        for p in self.base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)      # small random
        self.B = nn.Parameter(torch.zeros(base.out_features, r))            # zero, so the adapter starts as an exact no-op
    def forward(self, x): return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)
    def merge(self):                                                         # fold the adapter into the weight for serving
        with torch.no_grad(): self.base.weight += self.scale * (self.B @ self.A)
        return self.base
lin = nn.Linear(512, 512); l = LoRALinear(lin, r=8)
x = torch.randn(4, 512)
print("adapter is a no-op at start:", torch.allclose(l(x), lin(x)))
full, lora = 512 * 512 + 512, 8 * 512 + 512 * 8
print("one 512x512 layer: full fine-tune %d parameters, LoRA r=8 trains %d (%.1f%%)" % (full, lora, 100 * lora / full))
with torch.no_grad(): l.B.normal_(0, 0.02)
y_adapter = l(x); merged = copy.deepcopy(l).merge(); print("merged weights reproduce the adapter output:", torch.allclose(y_adapter, merged(x), atol=1e-5))
print("LoRA parameter counts for a 7B-class model (hidden 4096, 32 layers, adapters on q,k,v,o): r=8 -> %.1fM, r=16 -> %.1fM, r=64 -> %.1fM" % tuple(4 * 32 * 2 * 4096 * r / 1e6 for r in (8, 16, 64)))

## 2. does low rank suffice? pretrain on digits, adapt to a shifted 'camera', compare methods

In [ ]:
X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)
rs = np.random.RandomState(1); pattern = (rs.rand(1, 64) > 0.5).astype(np.float32)
shift = lambda a: np.clip(a * 0.55 + 0.35 * pattern + 0.1, 0, 1).astype(np.float32)
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr_s, Xte_s = T(shift(Xtr)), T(shift(Xte)); Xtr_t, Xte_t = T(Xtr), T(Xte); ytr_t, yte_t = torch.tensor(ytr), torch.tensor(yte)
def make(): return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
def fit(model, Xa, ya, epochs, lr=2e-3, params=None):
    o = torch.optim.Adam(params if params is not None else [p for p in model.parameters() if p.requires_grad], lr=lr)
    for _ in range(epochs):
        perm = torch.randperm(len(Xa))
        for i in range(0, len(perm), 64):
            b = perm[i:i + 64]; o.zero_grad(); F.cross_entropy(model(Xa[b]), ya[b]).backward(); o.step()
    return model
acc = lambda m, a: (m(a).argmax(1) == yte_t).float().mean().item()
torch.manual_seed(0); base = fit(make(), Xtr_t, ytr_t, 40)
print("\npretrained: old domain %.3f | shifted domain %.3f" % (acc(base, Xte_t), acc(base, Xte_s)))
n_all = sum(p.numel() for p in base.parameters())
# few-shot adaptation: only 100 labelled shifted examples
idx = np.random.RandomState(0).choice(len(Xtr_s), 100, replace=False); Xa, ya = Xtr_s[idx], ytr_t[idx]
def with_lora(r):
    m = copy.deepcopy(base)
    for i in (0, 2, 4): m[i] = LoRALinear(m[i], r=r, alpha=2 * r)
    return m
print("method                      trainable params   shifted acc   old-domain acc")
rows = []
m = copy.deepcopy(base); fit(m, Xa, ya, 60, lr=5e-4); rows.append(("full fine-tune", n_all, acc(m, Xte_s), acc(m, Xte_t)))
m = copy.deepcopy(base)
for p in m.parameters(): p.requires_grad_(False)
for p in m[4].parameters(): p.requires_grad_(True)
fit(m, Xa, ya, 60, lr=2e-3); rows.append(("last layer only", sum(p.numel() for p in m[4].parameters()), acc(m, Xte_s), acc(m, Xte_t)))
for r in (1, 2, 4, 8, 16):
    torch.manual_seed(0); m = with_lora(r); fit(m, Xa, ya, 60, lr=5e-3)
    rows.append(("LoRA r=%d (all 3 layers)" % r, sum(p.numel() for p in m.parameters() if p.requires_grad), acc(m, Xte_s), acc(m, Xte_t)))
for name, npar, a_s, a_o in rows: print("%-27s %10d       %.3f         %.3f" % (name, npar, a_s, a_o))

## 3. why low rank works here: the fine-tuning update is itself low-rank-ish

In [ ]:
m = copy.deepcopy(base); fit(m, Xtr_s, ytr_t, 30, lr=5e-4)
delta = (m[2].weight - base[2].weight).detach()
sv = torch.linalg.svdvals(delta); energy = (sv ** 2).cumsum(0) / (sv ** 2).sum()
print("\nfull fine-tune update of the 256x256 layer: top-1 / top-4 / top-16 singular values hold %.0f%% / %.0f%% / %.0f%% of the energy" % tuple(100 * energy[k - 1].item() for k in (1, 4, 16)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.